![Diagram](./images/01_img.png)

# 🩺 Appointment Intent Router with Ollama

### Run a fine-tuned scheduling model locally — no cloud, no API keys

This tutorial walks through serving a **fine-tuned intent classifier** entirely on your own machine with [Ollama](https://ollama.com). The model reads a message sent to a doctor-appointment booking assistant and returns **exactly one routing label** that tells your app what to do next.

The model was fine-tuned from `meta-llama/Llama-3.2-1B-Instruct` using **LoRA supervised fine-tuning (SFT)**, then exported to quantized **GGUF** for Ollama. Its job is intentionally narrow: it is a *router*, not a chatbot.


> **📌 Prerequisites**
> - [Ollama](https://ollama.com/download) installed and running locally
> - Python managed with [uv](https://docs.astral.sh/uv/) (see Step 1)
> - ~1 GB of disk space for the quantized model

> **📚 Ollama reference docs**
> - [API introduction](https://docs.ollama.com/api/introduction) · [Chat API](https://docs.ollama.com/api/chat) · [Generate API](https://docs.ollama.com/api/generate) · [Modelfile reference](https://docs.ollama.com/modelfile)


## What was fine-tuned?

A small instruction model was fine-tuned for **intent classification** in a doctor-appointment booking system.

| Component | Value |
|---|---|
| **Base model** | `meta-llama/Llama-3.2-1B-Instruct` |
| **Fine-tuning method** | LoRA supervised fine-tuning (SFT) |
| **Deployment format** | Quantized GGUF for Ollama |
| **Local runtime** | Ollama on `http://localhost:11434` |
| **Output contract** | One exact intent label, no explanation |


> **💡 Key idea**
> This model is **not a general chatbot — it is a router.** In a real app, its output decides which function, workflow, or human queue should handle the user's message.


## Why create a local Ollama wrapper?

The Hugging Face GGUF contains the fine-tuned weights — but weights alone aren't enough. The model still needs **the exact prompt format it saw during training.**

> **⚠️ Without the right prompt format**
> If you call the raw GGUF without the matching system prompt and chat template, it may produce full sentences instead of clean intent labels.

The local Ollama wrapper fixes this by baking the inference contract directly into the model:

| What we bake in | Why |
|---|---|
| 🧭 System instruction | Tells the model it's a router and lists the valid labels |
| 🌡️ Deterministic decoding (`temperature=0`) | Same input → same label, every time |
| 🧩 Llama 3.2 chat template | Matches the format used during fine-tuning |
| 🛑 Stop sequences | Cuts generation at the end of the label |
| ✂️ Short output length (`num_predict=16`) | A label is only a few tokens |

The result is a clean, reusable local model name: **`appointment-intent-sft`**.


## Step 1 · Dependencies

Dependencies are managed with [uv](https://docs.astral.sh/uv/) and pinned in `pyproject.toml` / `uv.lock`. Set up the project and launch the notebook with:

```bash
uv sync
uv run jupyter lab Appointment_Intent_Router_with_Ollama.ipynb
```

Then select the project's `.venv` kernel.

> **📝 Note**
> No in-notebook `pip install` is required. This notebook uses only **local Ollama** plus a few Python helpers — `requests`, `pandas`, `scikit-learn`, `langchain`, `langchain-ollama`, and `pydantic`. It does **not** need Hugging Face, PyTorch, or OpenAI.


## Step 2 · Configure the Ollama model

The cell below defines everything the rest of the notebook depends on:

- **`SOURCE_MODEL`** — the GGUF model published on Hugging Face for Ollama to pull.
- **`LOCAL_MODEL`** — the friendly local name we'll create with the correct routing prompt and template.
- **`INTENTS` / `SYSTEM_PROMPT`** — the routing contract: the valid labels and the instruction that enforces them.
- **`LLAMA32_TEMPLATE`** — the Llama 3.2 chat template that matches fine-tuning.

> **💡 Tip**
> The Ollama base URL is read from the `OLLAMA_BASE_URL` environment variable, falling back to `http://localhost:11434`. Override it if Ollama runs on another host or port.


![Diagram](./images/02_img.png)

In [1]:
import json
import os
import time
from typing import Dict

import pandas as pd
import requests
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

OLLAMA_BASE_URL = os.getenv("OLLAMA_BASE_URL", "http://localhost:11434").rstrip("/")
SOURCE_MODEL = "hf.co/vidhyakshayakannan/appointment-intent-llama32-1b-sft-GGUF:Q4_K_M"
LOCAL_MODEL = "appointment-intent-sft"

INTENTS = [
    "LIST_APPOINTMENTS",
    "CANCEL_APPOINTMENTS",
    "RESCHEDULE_APPOINTMENTS",
    "UNSUPPORTED_REQUEST",
    "BLOCK_SLOTS",
    "UNBLOCK_SLOTS",
    "UPDATE_SPECIAL_SLOTS",
    "CONVERSATIONAL_GREETING",
    "CLOSING_CONVERSATION",
]
VALID_INTENTS = set(INTENTS)

SYSTEM_PROMPT = """You are an intent classifier for a doctor appointment booking system.
Classify the user message into exactly one of these following intents:
LIST_APPOINTMENTS, CANCEL_APPOINTMENTS, RESCHEDULE_APPOINTMENTS, UNSUPPORTED_REQUEST, BLOCK_SLOTS, UNBLOCK_SLOTS, UPDATE_SPECIAL_SLOTS, CONVERSATIONAL_GREETING, CLOSING_CONVERSATION
Reply with ONLY the EXACT intent label. Nothing else."""

LLAMA32_TEMPLATE = """<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 27 Jun 2026

{{ .System }}<|eot_id|><|start_header_id|>user<|end_header_id|>

{{ .Prompt }}<|eot_id|><|start_header_id|>assistant<|end_header_id|>

"""

print(f"Ollama API:   {OLLAMA_BASE_URL}")
print(f"Source model: {SOURCE_MODEL}")
print(f"Local model:  {LOCAL_MODEL}")


Ollama API:   http://localhost:11434
Source model: hf.co/vidhyakshayakannan/appointment-intent-llama32-1b-sft-GGUF:Q4_K_M
Local model:  appointment-intent-sft


## Step 3 · Check that Ollama is running

Every API call below talks to a running Ollama server, so we verify the connection first by hitting `GET /api/tags` (which also lists installed models).

> **⚠️ If this cell fails**
> Start the server, then re-run the cell:
> - Open the **Ollama desktop app**, *or*
> - run **`ollama serve`** in a terminal.


In [2]:
def ollama_get(path: str, timeout: int = 10) -> Dict:
    response = requests.get(f"{OLLAMA_BASE_URL}{path}", timeout=timeout)
    response.raise_for_status()
    return response.json()

try:
    tags = ollama_get("/api/tags")
    print("Ollama is running")
    print("Installed models:")
    for model in tags.get("models", []):
        size_gb = model.get("size", 0) / 1e9
        print(f"- {model.get('name')} ({size_gb:.2f} GB)")
except requests.exceptions.ConnectionError as exc:
    raise RuntimeError(
        "Could not connect to Ollama. Start Ollama first, then rerun this cell."
    ) from exc


Ollama is running
Installed models:
- appointment-intent-sft:latest (0.81 GB)
- hf.co/vidhyakshayakannan/appointment-intent-llama32-1b-sft-GGUF:Q4_K_M (0.81 GB)


## Step 4 · Pull the fine-tuned GGUF model

This downloads the model with `POST /api/pull`. The helper streams progress events, prints download percentage, and skips the download entirely if the model is already installed.

> **📝 Note**
> The quantized model is about **800 MB**, so the first pull can take a few minutes depending on your network. Subsequent runs are instant.


In [3]:
def installed_model_names() -> set:
    data = ollama_get("/api/tags")
    return {model["name"] for model in data.get("models", [])}


def pull_model(model_name: str) -> None:
    if model_name in installed_model_names():
        print(f"Already installed: {model_name}")
        return

    print(f"Pulling {model_name}...")
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/pull",
        json={"model": model_name, "stream": True},
        stream=True,
        timeout=None,
    )
    response.raise_for_status()

    last_message = None
    for line in response.iter_lines():
        if not line:
            continue
        event = json.loads(line.decode("utf-8"))
        if event.get("error"):
            raise RuntimeError(event["error"])

        status = event.get("status", "")
        completed = event.get("completed")
        total = event.get("total")

        if completed and total:
            message = f"{status}: {completed / total:.0%}"
        else:
            message = status

        if message and message != last_message:
            print(message)
            last_message = message

    print("Pull complete")


pull_model(SOURCE_MODEL)


Already installed: hf.co/vidhyakshayakannan/appointment-intent-llama32-1b-sft-GGUF:Q4_K_M


## Step 5 · Create the local routing model wrapper

This calls `POST /api/create` to build `appointment-intent-sft` on top of the downloaded GGUF, attaching the routing system prompt, the Llama 3.2 template, and deterministic parameters.

> **💡 Why this matters**
> The wrapper gives Ollama the **same routing instructions used during fine-tuning** and makes inference deterministic (`temperature=0`, output capped to a short label). This is what turns raw weights into a reliable, reusable router.


![Diagram](./images/03_img.png)

In [4]:
def create_local_router_model(local_model: str = LOCAL_MODEL, source_model: str = SOURCE_MODEL) -> None:
    payload = {
        "model": local_model,
        "from": source_model,
        "system": SYSTEM_PROMPT,
        "template": LLAMA32_TEMPLATE,
        "parameters": {
            "temperature": 0,
            "num_predict": 16,
            "stop": ["<|start_header_id|>", "<|end_header_id|>", "<|eot_id|>"],
        },
        "stream": False,
    }

    response = requests.post(f"{OLLAMA_BASE_URL}/api/create", json=payload, timeout=None)
    response.raise_for_status()
    data = response.json()
    if data.get("status") != "success":
        print(data)
    else:
        print(f"Created local model: {local_model}")


create_local_router_model()


Created local model: appointment-intent-sft


## Step 6 · Inspect the local model

A quick sanity check with `POST /api/show`: confirm the wrapper exists and that Ollama is using the system prompt and parameters we configured in Step 5.


In [5]:
def show_model(model_name: str = LOCAL_MODEL) -> Dict:
    response = requests.post(
        f"{OLLAMA_BASE_URL}/api/show",
        json={"model": model_name},
        timeout=60,
    )
    response.raise_for_status()
    return response.json()

model_info = show_model()
print("Model:", LOCAL_MODEL)
print("Details:", model_info.get("details", {}))
print("\nParameters:\n", model_info.get("parameters", "").strip())
print("\nSystem prompt:\n", model_info.get("system", "").strip())


Model: appointment-intent-sft
Details: {'parent_model': 'hf.co/vidhyakshayakannan/appointment-intent-llama32-1b-sft-GGUF:Q4_K_M', 'format': 'gguf', 'family': 'llama', 'families': ['llama'], 'parameter_size': '1.2B', 'quantization_level': 'Q4_K_M'}

Parameters:
 stop                           "<|start_header_id|>"
stop                           "<|end_header_id|>"
stop                           "<|eot_id|>"
temperature                    0
num_predict                    16

System prompt:
 You are an intent classifier for a doctor appointment booking system.
Classify the user message into exactly one of these following intents:
LIST_APPOINTMENTS, CANCEL_APPOINTMENTS, RESCHEDULE_APPOINTMENTS, UNSUPPORTED_REQUEST, BLOCK_SLOTS, UNBLOCK_SLOTS, UPDATE_SPECIAL_SLOTS, CONVERSATIONAL_GREETING, CLOSING_CONVERSATION
Reply with ONLY the EXACT intent label. Nothing else.


## Step 7 · Define the local inference helper

This is the core function you'd call inside a local scheduling app. It sends one user message to `POST /api/chat`, normalizes the model's raw output to an official label, and returns useful metadata.

| In | Out |
|---|---|
| Raw user message (string) | `intent` — normalized routing label |
| | `raw_output` — exactly what the model emitted |
| | `valid_intent` — whether it maps to a known label |
| | `latency_s`, `prompt_tokens`, `output_tokens` |

> **💡 Tip**
> `normalize_intent` defends against small output quirks — stray whitespace, punctuation, or a trained synonym like `IRRELEVANT_QUERY` — so the app always routes on a clean, official label while `raw_output` stays visible for debugging.


In [6]:
INTENT_ALIASES = {
    # The trained model occasionally uses this natural synonym for out-of-scope requests.
    # We keep raw_output visible, but normalize the app-facing route to the official label.
    "IRRELEVANT_QUERY": "UNSUPPORTED_REQUEST",
    "UNSUPPORTED": "UNSUPPORTED_REQUEST",
    "UNSUPPORTED_QUERY": "UNSUPPORTED_REQUEST",
}


def normalize_intent(raw_text: str) -> str:
    text = raw_text.strip().upper().strip(" .:;`'")
    if text in INTENT_ALIASES:
        return INTENT_ALIASES[text]
    if text in VALID_INTENTS:
        return text

    # Small safety cleanup in case the model returns extra whitespace or punctuation.
    for intent in INTENTS:
        if intent in text:
            return intent
    for alias, official_intent in INTENT_ALIASES.items():
        if alias in text:
            return official_intent
    return text


def classify_appointment_message(message: str, model_name: str = LOCAL_MODEL) -> Dict:
    payload = {
        "model": model_name,
        "messages": [{"role": "user", "content": message}],
        "stream": False,
        "options": {
            "temperature": 0,
            "num_predict": 16,
        },
    }

    start = time.perf_counter()
    response = requests.post(f"{OLLAMA_BASE_URL}/api/chat", json=payload, timeout=120)
    response.raise_for_status()
    data = response.json()
    latency_s = time.perf_counter() - start

    raw_output = data.get("message", {}).get("content", "").strip()
    intent = normalize_intent(raw_output)

    return {
        "message": message,
        "intent": intent,
        "raw_output": raw_output,
        "valid_intent": intent in VALID_INTENTS,
        "latency_s": latency_s,
        "prompt_tokens": data.get("prompt_eval_count"),
        "output_tokens": data.get("eval_count"),
    }


classify_appointment_message("Can you move my appointment from Friday to next Monday?")


{'message': 'Can you move my appointment from Friday to next Monday?',
 'intent': 'RESCHEDULE_APPOINTMENTS',
 'raw_output': 'RESCHEDULE_APPOINTMENTS',
 'valid_intent': True,
 'latency_s': 1.4216252920450643,
 'prompt_tokens': 132,
 'output_tokens': 6}

## Step 8 · Evaluate against the golden dataset

`golden_dataset.csv` holds **300 hand-labelled `(USER_INPUT, TARGET_INTENT)` pairs**. We run every input through `classify_appointment_message`, compare the predicted route to the gold label, and report **precision, recall, and F1** per intent plus macro/weighted averages.


![Diagram](./images/04_img.png)

In [7]:
# Load the hand-labelled evaluation set
golden = pd.read_csv("golden_dataset.csv")

print(f"Loaded {len(golden)} labelled test cases from golden_dataset.csv\n")
print("Gold label distribution:")
print(golden["TARGET_INTENT"].value_counts().to_string())
golden.head()

Loaded 301 labelled test cases from golden_dataset.csv

Gold label distribution:
TARGET_INTENT
BLOCK_SLOTS                59
RESCHEDULE_APPOINTMENTS    51
LIST_APPOINTMENTS          46
UNBLOCK_SLOTS              40
UPDATE_SPECIAL_SLOTS       34
CANCEL_APPOINTMENTS        32
UNSUPPORTED_REQUEST        19
IRRELEVANT_QUERY            8
CONVERSATIONAL_GREETING     6
CLOSING_CONVERSATION        6


,TEST_ID,USER_INPUT,TARGET_INTENT
0,TEST_001,List my schedule for this afternoon.,LIST_APPOINTMENTS
1,TEST_002,What are my appointments Tuesday morning?,LIST_APPOINTMENTS
2,TEST_003,What's on my calendar this week?,LIST_APPOINTMENTS
3,TEST_004,What is my availability Monday morning?,LIST_APPOINTMENTS
4,TEST_005,View my schedule New Years Eve,LIST_APPOINTMENTS


### 8a · Run the router over every test case

This calls the local model once per row (~300 calls). With deterministic decoding (`temperature=0`) the run is reproducible; progress prints every 25 cases.

> **💡 Tip**
> After the first warm-up call, each classification typically completes in a fraction of a second — a benefit of the tiny 1B model and short output length.


In [8]:
records = []
total = len(golden)

for i, row in enumerate(golden.itertuples(index=False), start=1):
    result = classify_appointment_message(row.USER_INPUT)
    records.append({
        "test_id": row.TEST_ID,
        "user_input": row.USER_INPUT,
        # Normalize BOTH sides so the eval reflects the route the app would actually take
        # (e.g. gold IRRELEVANT_QUERY -> UNSUPPORTED_REQUEST).
        "target_intent": normalize_intent(row.TARGET_INTENT),
        "predicted_intent": result["intent"],
        "raw_output": result["raw_output"],
        "latency_s": result["latency_s"],
    })
    if i % 25 == 0 or i == total:
        print(f"Classified {i}/{total}")

results = pd.DataFrame(records)
results["correct"] = results["target_intent"] == results["predicted_intent"]

print(f"\nOverall accuracy : {results['correct'].mean():.1%}")
print(f"Mean latency     : {results['latency_s'].mean():.2f}s per query")
results.head()

Classified 25/301
Classified 50/301
Classified 75/301
Classified 100/301
Classified 125/301
Classified 150/301
Classified 175/301
Classified 200/301
Classified 225/301
Classified 250/301
Classified 275/301
Classified 300/301
Classified 301/301

Overall accuracy : 84.1%
Mean latency     : 0.21s per query


,test_id,user_input,target_intent,predicted_intent,raw_output,latency_s,correct
0,TEST_001,List my schedule for this afternoon.,LIST_APPOINTMENTS,LIST_APPOINTMENTS,LIST_APPOINTMENTS,0.292348,True
1,TEST_002,What are my appointments Tuesday morning?,LIST_APPOINTMENTS,LIST_APPOINTMENTS,LIST_APPOINTMENTS,0.205178,True
2,TEST_003,What's on my calendar this week?,LIST_APPOINTMENTS,LIST_APPOINTMENTS,LIST_APPOINTMENTS,0.195432,True
3,TEST_004,What is my availability Monday morning?,LIST_APPOINTMENTS,LIST_APPOINTMENTS,LIST_APPOINTMENTS,0.190892,True
4,TEST_005,View my schedule New Years Eve,LIST_APPOINTMENTS,LIST_APPOINTMENTS,LIST_APPOINTMENTS,0.191353,True


### 8b · Precision, recall, and F1

For a multi-class router, each intent gets its own scores:

| Metric | Question it answers | Low score means |
|---|---|---|
| **Precision** | Of messages routed *to* an intent, how many belonged there? | False alarms |
| **Recall** | Of messages that *belonged to* an intent, how many were caught? | Misses |
| **F1** | Harmonic mean of precision and recall | Weak on either side |

> **📊 Reading the averages**
> The **macro average** weights every intent equally — the most honest single-number summary for a router, because rare intents matter as much as common ones. The **weighted average** weights by support and tracks overall accuracy more closely.


In [9]:
# Per-intent precision/recall/F1 plus macro & weighted averages, in one call
print(classification_report(
    results["target_intent"],
    results["predicted_intent"],
    zero_division=0,
))

                         precision    recall  f1-score   support

            BLOCK_SLOTS       0.98      0.80      0.88        59
    CANCEL_APPOINTMENTS       0.90      0.59      0.72        32
   CLOSING_CONVERSATION       1.00      0.83      0.91         6
CONVERSATIONAL_GREETING       0.86      1.00      0.92         6
      LIST_APPOINTMENTS       0.96      0.96      0.96        46
RESCHEDULE_APPOINTMENTS       0.92      0.94      0.93        51
          UNBLOCK_SLOTS       0.97      0.78      0.86        40
    UNSUPPORTED_REQUEST       0.44      0.74      0.56        27
   UPDATE_SPECIAL_SLOTS       0.73      0.97      0.84        34

               accuracy                           0.84       301
              macro avg       0.86      0.85      0.84       301
           weighted avg       0.88      0.84      0.85       301

